In [1]:
import os, shlex, subprocess, json
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple, Dict
from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")

#from data.minbpe import BasicTokenizer as Tokenizer
from src.minbpe import RegexTokenizer as Tokenizer
#from src.gpt import GPTLanguageModel
from src.transformer.model_relative_positional_encoding import GPTLanguageModel

import pandas as pd
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
def now():
    return datetime.now().astimezone().strftime('%FT%T%:z')

def check_ckpt_files(ckpt_dir, match="checkpoint_*.pt"):
    result = {}

    def handle(pt_file):
        #tags = str(pt_file).replace("checkpoint_", "").replace(".pt", "").split("-", 1)
        #epoch, step = tags[0], int(tags[1])
        jf = Path(str(pt_file).replace(".pt", ".json"))
        d = json.loads(jf.read_text())
        epoch = d['epoch']

        if epoch not in result:
            result[epoch] = []

        result[epoch].append((pt_file, d['step'], d['train_loss']))

    for f in ckpt_dir.glob(match):
        handle(f)

    #key=lambda x: x.stat().st_ctime,
    for _, v in result.items():
        keep = [
            sorted(v, key=lambda x: x[1], reverse=True)[0][0],  # the last step of the epoch 
            sorted(v, key=lambda x: x[2], reverse=False)[0][0], # the lowest train loss of the epoch
        ]

        for e in v:
            if e[0] not in keep:
                print(f"Remove ckpt file: {e[0]}")
                os.remove(e[0])

def send_notification(title, message):
    cmd = os.getenv("send_notification")
    if cmd is None:
        return

    command = shlex.split(cmd)
    command.append(title)
    command.append(message)

    _ = subprocess.Popen(command)


In [3]:
def print_model_structure(model: nn.Module, indent: str = '') -> None:
    """
    Custom function to print model structure in a hierarchical format
    """
    for name, child in model.named_children():
        params = sum(p.numel() for p in child.parameters())
        print(f"{indent}├─ {name}: {child.__class__.__name__} ({params:,} parameters)")
        print_model_structure(child, indent + '│  ')

def get_model_stats(model: torch.nn.Module) -> pd.DataFrame:
    """
    Create a DataFrame with detailed layer statistics
    """
    stats = []
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:  # Only leaf modules
            params = sum(p.numel() for p in module.parameters())
            trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)

            stats.append({
                'Layer Name': name, 'Type': module.__class__.__name__,
                'Parameters': params, 'Trainable': trainable,
            })

    return pd.DataFrame(stats)

class TextDataset(Dataset):
    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[index:index + self.block_size]
        y = self.data[index + 1:index + self.block_size + 1]
        return x, y

def get_dataloaders(
        train_data: torch.Tensor,
        val_data: torch.Tensor,
        block_size: int,
        batch_size: int,
        device: torch.device,
) -> Tuple[DataLoader, DataLoader]:
    train_dataset = TextDataset(train_data.to(device), block_size)
    val_dataset = TextDataset(val_data.to(device), block_size)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_loader, val_loader

@torch.no_grad()
def estimate_loss(model: torch.nn.Module, eval_dataset: Dict[str, DataLoader]) -> Dict[str, float]:
    output = {}
    model.eval()

    for split, loader in eval_dataset.items():
        losses = torch.zeros(eval_batches)
        for i, (x, y) in enumerate(loader):
            with torch.no_grad():
                _, loss = model(x, y)
            losses[i] = loss.item()

        output[split] = float(losses.mean().item())

    model.train()
    return output

In [4]:
#### 1. init
run_name = "ch011_train"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer_dir = Path("data") / "tokenizer"
checkpoint_dir = Path("data") / "ch11"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

In [5]:
#### 2. setup
tokenizer = Tokenizer()
tokenizer.load(model_file=str(tokenizer_dir / "tokenizer.model"))

train_data = torch.load(tokenizer_dir / 'train.tokens.pt')
val_data = torch.load(tokenizer_dir / 'validation.tokens.pt')

print(f"--> train_data: {train_data.size()[0]:_}, val_data: {val_data.size()[0]:_}")

--> train_data: 12_092_088, val_data: 616_680


In [6]:
#### 3. parameters
##### 3.1 model parameters
parameters = {
    'vocab_size': len(tokenizer.vocab),
    'n_embd': 512,
    'block_size': 256,
    'n_head': 8,
    'n_layer': 4,
    'dropout': 0.2,
}

##### 3.2 training parameters
total_epoches = 12
eval_interval = 1_000
eval_batches = 1_000 # 5_000

batch_size = 64 # 32， 64， 96
base_lr = 5e-4
min_lr = 5e-6

In [7]:
#### 4. datasets
train_loader, val_loader = get_dataloaders(
    train_data=train_data,
    val_data=val_data,
    block_size=parameters['block_size'],
    batch_size=batch_size,
    device=device,
)

eval_batches = min(eval_batches, len(val_loader))
eval_dataset = {}

eval_dataset['train'], eval_dataset['val'] = get_dataloaders(
    train_data=train_data[:eval_batches*batch_size],
    val_data=val_data[:eval_batches*batch_size],
    block_size=parameters['block_size'],
    batch_size=batch_size,
    device=device,
)

print(f"{now()} train_batches={len(train_loader):_}, validation_batches={len(val_loader):_}, eval_batches={eval_batches:_}")

2025-09-28T09:46:56%:z train_batches=188_935, validation_batches=9_632, eval_batches=1_000


In [8]:
#### 5. Scheduler calculation
last_epoch, last_step = 1, 0
epoch_steps = len(train_loader)
total_steps = epoch_steps * total_epoches

gradient_accumulation_steps = 10
scheduler_interval = 20_000 # (0.1 * total_steps) // gradient accumulation steps

print(f"{now()} epoch_steps={epoch_steps:_}, total_steps={total_steps:_}, scheduler_interval={scheduler_interval:_}")

2025-09-28T09:46:56%:z epoch_steps=188_935, total_steps=2_267_220, scheduler_interval=20_000


In [9]:
#### 6. llm
model = GPTLanguageModel(
    vocab_size=parameters['vocab_size'],
    n_embd=parameters['n_embd'],
    block_size=parameters['block_size'],
    n_head=parameters['n_head'],
    n_layer=parameters['n_layer'],
    dropout=parameters['dropout'],
    device=device,
).to(device)

model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr)

warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=scheduler_interval,
)

#scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#    optimizer=optimizer,
#    T_max=2*scheduler_interval,
#    eta_min=min_learning_rate,
#)

cawr = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=scheduler_interval*2,
    T_mult=1,
    eta_min=min_lr,
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup, cawr],
    milestones=[scheduler_interval],
)

parameters_m = sum(p.numel() for p in model.parameters())/1e6
print(f'Model parameters: {parameters_m:.3f}M')

Model parameters: 13.660M


In [10]:
#### 7. last checkpoints
#key=lambda x: x.stat().st_ctime
get_step = lambda x: int(x.name.replace("checkpoint_", "").replace(".pt", "").split("-")[-1])
ckpt_files = sorted(checkpoint_dir.glob("checkpoint_*.pt"), key=get_step, reverse=True)

if len(ckpt_files) > 0:
    checkpoint_path = ckpt_files[0]
    last_ckpt = torch.load(checkpoint_path, map_location=device) # weights_only=True
    last_epoch = last_ckpt['meta']['epoch']
    last_step = last_ckpt['meta']['step']
    if last_step % len(train_loader) == 0:
        last_epoch += 1

    print(f"load: last_checkpoint={checkpoint_path}, last_step={last_step:07_}")    
    model.load_state_dict(last_ckpt['model_state_dict'])
    optimizer.load_state_dict(last_ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(last_ckpt['scheduler_state_dict'])

load: last_checkpoint=data/ch11/checkpoint_010-1889350.pt, last_step=1_889_350


In [11]:
#for module in model.modules():
#    if isinstance(module, nn.Dropout):
#        module.p = 0.2

#scheduler._schedulers[1].T_mult = 1

print(f"==> optimizer: {optimizer}")

for v in scheduler._schedulers:
    print(f"==> scheduler:\n{v.state_dict()}")

==> optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0005
    lr: 0.0004415186041396605
    maximize: False
    weight_decay: 0.01
)
==> scheduler:
{'start_factor': 0.01, 'end_factor': 1.0, 'total_iters': 20000, 'base_lrs': [0.0005], 'last_epoch': 19999, '_step_count': 20000, '_is_initial': False, '_get_lr_called_within_step': False, '_last_lr': [0.0004999752500000193]}
==> scheduler:
{'T_0': 40000, 'T_i': 40000, 'T_mult': 1, 'eta_min': 5e-06, 'T_cur': 8935, 'base_lrs': [0.0005], 'last_epoch': 168935, '_step_count': 0, '_is_initial': False, '_get_lr_called_within_step': False, '_last_lr': [0.0004415186041396605]}


In [12]:
#### 8. estimate losses
def estimate_and_save(epoch, step):
    t0 = datetime.now()
    losses = estimate_loss(model, eval_dataset)
    #current_lr = scheduler.get_last_lr()[0]
    current_lr = optimizer.param_groups[0]['lr']

    # Save checkpoint
    prefix = str(checkpoint_dir / f"checkpoint_{epoch:03d}-{step:06d}")

    meta = {
        'created_at': now(),
        'run_name': run_name,
        'parameters': parameters,

        'epoch': epoch,
        'step': step,
        'learning_rate': current_lr,
        'train_loss': losses['train'],
        'val_loss': losses['val'],

        'dataset': { 'batch_size': batch_size, 'train': train_data.size(), 'val': val_data.size() }
    }

    checkpoint = {
        'meta': meta,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }

    with open(prefix + ".json", 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
        f.write("\n")

    torch.save(checkpoint, prefix+".pt")
    check_ckpt_files(checkpoint_dir)

    elapsed = datetime.now() - t0

    print(
        f"\n{now()} epoch={epoch}/{total_epoches}, step={step:07_}/{total_steps:07_}, "
        f"train_loss={losses['train']:.3f}, val_loss={losses['val']:.3f}, "
        f"\n    learning_rate={current_lr:.6f}, checkpoint={prefix}.pt, elapsed={str(elapsed)}"
    )

    return losses

In [ ]:
#### 9. training
batch_processed = (last_epoch - 1) * len(train_loader)
#optimizer_processed = last_step // gradient_accumulation_steps
optimizer_processed = scheduler.last_epoch

print(
    f"{now()} Running: last_epoch={last_epoch}/{total_epoches}, "
    f"last_step={last_step:_}/{total_steps:_}, optimizer_processed={optimizer_processed:_}"
)

for epoch in range(last_epoch, total_epoches+1):
    last_step = last_step if epoch == last_epoch else batch_processed

    msg = "{} starting epoch: epoch={}/{}, last_step={:_}/{:_}".format(
        now(), epoch, total_epoches, last_step, total_steps,
    )

    print(msg)
    send_notification(run_name, msg)

    for idx, (x_batch, y_batch) in enumerate(train_loader):
        batch_processed += 1
        if batch_processed <= last_step:
            continue

        # a. Training
        logits, loss = model(x_batch, y_batch)
        loss /= gradient_accumulation_steps
        loss.backward()
        #batch_loss = loss.item()

        # b. Optimization
        if batch_processed % gradient_accumulation_steps != 0:
            continue

        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_processed += 1
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        # c. Evalation
        if optimizer_processed % eval_interval != 0:
            continue

        # current_lr = scheduler.get_last_lr()[0]
        current_lr = optimizer.param_groups[0]['lr']
        losses = estimate_and_save(epoch, step=batch_processed)
        epoch_progress = f"{(idx+1) / epoch_steps:.3f}"
        total_progress = f"{batch_processed:_}/{total_steps:_}"

        msg = f"{now()} evaluation: epoch={epoch}/{total_epoches}, " + \
            f"epoch_progress={epoch_progress}, total_progress={total_progress}, " + \
            f"optimizer_processed={optimizer_processed:_}, learning_rate={current_lr:.6f}, " + \
            f"train_loss={losses['train']:.3f}, validation_loss={losses['val']:.3f}"

        send_notification(run_name, msg)

    epoch_pt = checkpoint_dir / f"checkpoint_{epoch:03d}-{batch_processed:06d}.pt"
    if not epoch_pt.exists():
        current_lr = optimizer.param_groups[0]['lr']
        losses = estimate_and_save(epoch, step=batch_processed)

send_notification(run_name, "Finished")

2025-09-28T09:46:59%:z Running: last_epoch=11/12, last_step=1_889_350/2_267_220, optimizer_processed=188_935
2025-09-28T09:46:59%:z starting epoch: epoch=11/12, last_step=1_889_350/2_267_220
{"status":1,"request":"5b1507f9-092d-415e-8c46-1f9ec86e6037"}